# 08 - Synthetic proof: paired-partition evaluation leaks the label prior

Dataset-free, GPU-free demonstration that the leakage effect found in
`07_run_protocol_leak_analysis.py` on GIED is a **structural** property of
the paired Dirichlet train/test partitioning protocol, not an artifact of
endoscopy data.

A predictor that sees **no features at all** - only each client's training
label prior - should sit at chance (AUC = 0.5) under honest (pooled) scoring.
Under the paired protocol, the same predictor's AUC should rise above chance
as a function of Dirichlet alpha, purely because train and test shards for
the same client share a partition-induced label-prior correlation.

Cheap to run: no images, no training, so we can afford 50-100 seeds per
setting - fixing the seed-count weakness that the GIED-scale experiments
(CPU-bound, 5 seeds) can't avoid.

In [ ]:
from __future__ import annotations

import itertools
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from scipy import stats as sps

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR = Path("synthetic_leak_out")
(OUT_DIR / "csv").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)

print("output dir:", OUT_DIR.resolve())

## 1. Synthetic label generator

We only need **labels**, not features, for a prior-only control - the whole
point is that this predictor never looks at data. K classes, an optional
binary grouping (mirrors GIED's abnormal-vs-normal framing), and a global
class prior you can skew to mimic real imbalance.

In [ ]:
def make_synthetic_labels(n: int, n_classes: int, seed: int,
                           class_weights: np.ndarray | None = None) -> np.ndarray:
    rng = np.random.RandomState(seed)
    if class_weights is None:
        class_weights = np.ones(n_classes) / n_classes
    class_weights = class_weights / class_weights.sum()
    return rng.choice(n_classes, size=n, p=class_weights)


def binarize(y: np.ndarray, positive_classes: list[int]) -> np.ndarray:
    return np.isin(y, positive_classes).astype(int)

## 2. Dirichlet partitioning (pooled vs paired)

Mirrors `data_utils.dirichlet_partition` / `dirichlet_partition_paired`.
Re-implemented here standalone so this notebook has zero dependency on the
GIED source tree - swap these two functions out for the real imports if you
want a byte-for-byte reproduction of the project's partitioning logic.

In [ ]:
def dirichlet_partition(y: np.ndarray, n_clients: int, alpha: float,
                         seed: int) -> list[np.ndarray]:
    """Standard non-IID label-skew partition of one label vector."""
    rng = np.random.RandomState(seed)
    n_classes = int(y.max()) + 1
    client_idx = [[] for _ in range(n_clients)]
    if np.isinf(alpha):
        # IID: shuffle and split evenly
        order = rng.permutation(len(y))
        for i, idx in enumerate(np.array_split(order, n_clients)):
            client_idx[i] = idx
        return [np.asarray(c) for c in client_idx]

    for c in range(n_classes):
        idx_c = np.flatnonzero(y == c)
        rng.shuffle(idx_c)
        proportions = rng.dirichlet(np.repeat(alpha, n_clients))
        splits = (np.cumsum(proportions) * len(idx_c)).astype(int)[:-1]
        for i, part in enumerate(np.split(idx_c, splits)):
            client_idx[i].extend(part.tolist())
    return [np.asarray(sorted(c), dtype=int) for c in client_idx]


def dirichlet_partition_paired(train_y: np.ndarray, test_y: np.ndarray,
                                n_clients: int, alpha: float, seed: int):
    """Partition train AND test with the SAME proportion matrix per class -
    the practice this experiment is auditing. Returns (train_shards,
    test_shards, proportions).
    """
    rng = np.random.RandomState(seed)
    n_classes = int(max(train_y.max(), test_y.max())) + 1
    tr_idx = [[] for _ in range(n_clients)]
    te_idx = [[] for _ in range(n_clients)]
    proportions_all = {}

    for c in range(n_classes):
        if np.isinf(alpha):
            proportions = np.full(n_clients, 1.0 / n_clients)
        else:
            proportions = rng.dirichlet(np.repeat(alpha, n_clients))
        proportions_all[c] = proportions

        for y_arr, bucket in ((train_y, tr_idx), (test_y, te_idx)):
            idx_c = np.flatnonzero(y_arr == c)
            rng.shuffle(idx_c)
            splits = (np.cumsum(proportions) * len(idx_c)).astype(int)[:-1]
            for i, part in enumerate(np.split(idx_c, splits)):
                bucket[i].extend(part.tolist())

    tr_shards = [np.asarray(sorted(c), dtype=int) for c in tr_idx]
    te_shards = [np.asarray(sorted(c), dtype=int) for c in te_idx]
    return tr_shards, te_shards, proportions_all

## 3. Prior-only predictor + AUC under both protocols

No features, no model: each client's score is simply its training-set
positive-class rate. Under **pooled** scoring (test held out whole, scored
with one global prior) this must sit at chance by construction. Under
**paired** scoring (each test row scored with its own client's prior) it
leaks whatever client-level correlation the shared partition matrix induced.

In [ ]:
def prior_only_auc_pair(train_y_bin: np.ndarray, test_y_bin: np.ndarray,
                         train_y_full: np.ndarray, test_y_full: np.ndarray,
                         n_clients: int, alpha: float, seed: int,
                         positive_classes: list[int]) -> dict:
    rng = np.random.RandomState(seed)

    # --- pooled (honest) protocol --------------------------------------
    tr_shards_pool = dirichlet_partition(train_y_full, n_clients, alpha, seed)
    cnt = np.zeros(int(train_y_full.max()) + 1)
    for c in np.unique(train_y_full):
        cnt[c] = (train_y_full == c).sum()
    global_rate = cnt[positive_classes].sum() / cnt.sum()
    s_pool = np.full(len(test_y_bin), global_rate)
    s_pool = s_pool + rng.normal(0, 1e-9, len(s_pool))  # break exact ties

    # --- paired (leaky) protocol -----------------------------------------
    tr_shards_pair, te_shards_pair, _ = dirichlet_partition_paired(
        train_y_full, test_y_full, n_clients, alpha, seed)
    s_pair = np.zeros(len(test_y_bin))
    for cid, (tri, tei) in enumerate(zip(tr_shards_pair, te_shards_pair)):
        cnt_c = np.bincount(train_y_full[tri],
                            minlength=int(train_y_full.max()) + 1).astype(float)
        rate_c = cnt_c[positive_classes].sum() / max(cnt_c.sum(), 1)
        s_pair[tei] = rate_c
    s_pair = s_pair + rng.normal(0, 1e-9, len(s_pair))

    if len(np.unique(test_y_bin)) < 2:
        auc_pool = auc_pair = np.nan
    else:
        auc_pool = roc_auc_score(test_y_bin, s_pool)
        auc_pair = roc_auc_score(test_y_bin, s_pair)

    return {"auc_pooled": auc_pool, "auc_paired": auc_pair,
            "leak": (auc_pair - auc_pool) if np.isfinite(auc_pair) else np.nan}

## 4. Sweep: alpha x seed x (n_clients, n_classes)

No images, no training -> cheap enough for 50-100 seeds per cell. Adjust
`N_SEEDS` down if you want a faster smoke test first.

In [ ]:
N_TRAIN = 4000
N_TEST = 1200
N_SEEDS = 100          # cheap: pure numpy, no GPU/training. Lower for a smoke test.
ALPHA_GRID = [0.05, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0, float("inf")]
N_CLIENTS_GRID = [5]          # matches CFG.fed.n_clients; add e.g. [5, 10] to test generality
N_CLASSES_GRID = [5]          # matches GIED's 5-class taxonomy; add e.g. [2, 5, 10]


def run_sweep() -> pd.DataFrame:
    rows = []
    t0 = time.time()
    combos = list(itertools.product(N_CLASSES_GRID, N_CLIENTS_GRID, ALPHA_GRID))
    for ci, (n_classes, n_clients, alpha) in enumerate(combos, 1):
        positive_classes = list(range(n_classes // 2 + 1, n_classes))  # mirror ABNORMAL_CLASSES split
        if not positive_classes:
            positive_classes = [n_classes - 1]
        for seed in range(N_SEEDS):
            y_tr = make_synthetic_labels(N_TRAIN, n_classes, seed=1000 + seed)
            y_te = make_synthetic_labels(N_TEST, n_classes, seed=2000 + seed)
            yb_tr = binarize(y_tr, positive_classes)
            yb_te = binarize(y_te, positive_classes)

            res = prior_only_auc_pair(yb_tr, yb_te, y_tr, y_te,
                                       n_clients, alpha, seed, positive_classes)
            rows.append({
                "n_classes": n_classes, "n_clients": n_clients,
                "alpha": alpha,
                "alpha_num": 1e9 if np.isinf(alpha) else alpha,
                "seed": seed, **res,
            })
        print(f"[{ci}/{len(combos)}] n_classes={n_classes} n_clients={n_clients} "
              f"alpha={alpha} done ({time.time()-t0:.1f}s elapsed)")
    return pd.DataFrame(rows)


df = run_sweep()
df.to_csv(OUT_DIR / "csv" / "synthetic_leak_alpha_sweep.csv", index=False)
df.head()

## 5. Summary table + sanity checks

Expect: `auc_pooled` ~= 0.5 for every alpha (by construction - it's a
single global constant score). `auc_paired` should rise monotonically as
alpha decreases (more non-IID -> more client-level label-prior signal to
leak). `leak` = auc_paired - auc_pooled is the headline quantity.

In [ ]:
summ = (df.groupby(["n_classes", "n_clients", "alpha_num"], as_index=False)
        .agg(auc_pooled_mean=("auc_pooled", "mean"),
             auc_pooled_std=("auc_pooled", "std"),
             auc_paired_mean=("auc_paired", "mean"),
             auc_paired_std=("auc_paired", "std"),
             leak_mean=("leak", "mean"),
             leak_std=("leak", "std"),
             n=("leak", "count"))
        .sort_values(["n_classes", "n_clients", "alpha_num"]))
summ.to_csv(OUT_DIR / "csv" / "synthetic_leak_summary.csv", index=False)

# one-sample t-test: is mean leak significantly > 0 at each alpha? (n=100 seeds -> well powered)
sig_rows = []
for (nc, ncl, a), g in df.groupby(["n_classes", "n_clients", "alpha_num"]):
    vals = g["leak"].dropna().to_numpy()
    if len(vals) > 2 and np.std(vals) > 0:
        t, p = sps.ttest_1samp(vals, popmean=0.0, alternative="greater")
    else:
        t, p = np.nan, np.nan
    sig_rows.append({"n_classes": nc, "n_clients": ncl, "alpha_num": a,
                      "t_stat": t, "p_value": p, "n_seeds": len(vals)})
sig_df = pd.DataFrame(sig_rows).sort_values(["n_classes", "n_clients", "alpha_num"])
sig_df.to_csv(OUT_DIR / "csv" / "synthetic_leak_significance.csv", index=False)

summ

In [ ]:
sig_df

## 6. Figure: leak magnitude vs Dirichlet alpha

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))

ax = axes[0]
for (nc, ncl), g in summ.groupby(["n_classes", "n_clients"]):
    g = g.sort_values("alpha_num")
    ax.errorbar(range(len(g)), g["auc_paired_mean"], yerr=g["auc_paired_std"],
                marker="o", lw=2, capsize=3, label=f"paired (K={nc}, C={ncl})")
    ax.errorbar(range(len(g)), g["auc_pooled_mean"], yerr=g["auc_pooled_std"],
                marker="s", lw=2, capsize=3, label=f"pooled (K={nc}, C={ncl})")
ax.axhline(0.5, ls="--", c="k", lw=1.2)
g0 = summ[(summ.n_classes == N_CLASSES_GRID[0]) & (summ.n_clients == N_CLIENTS_GRID[0])].sort_values("alpha_num")
ax.set_xticks(range(len(g0)))
ax.set_xticklabels(["IID" if v >= 1e8 else f"{v:g}" for v in g0.alpha_num], rotation=45, fontsize=8)
ax.set_xlabel("Dirichlet concentration alpha")
ax.set_ylabel("Prior-only AUROC (no features)")
ax.set_title("(a) Pooled stays at chance; paired leaks", fontweight="bold")
ax.legend(frameon=False, fontsize=7)
ax.grid(alpha=0.3)

ax = axes[1]
for (nc, ncl), g in summ.groupby(["n_classes", "n_clients"]):
    g = g.sort_values("alpha_num")
    ax.errorbar(range(len(g)), g["leak_mean"], yerr=g["leak_std"],
                marker="o", lw=2, capsize=3, label=f"K={nc}, C={ncl}")
ax.axhline(0.0, ls="--", c="k", lw=1.2)
ax.set_xticks(range(len(g0)))
ax.set_xticklabels(["IID" if v >= 1e8 else f"{v:g}" for v in g0.alpha_num], rotation=45, fontsize=8)
ax.set_xlabel("Dirichlet concentration alpha")
ax.set_ylabel("Leak = AUC_paired - AUC_pooled")
ax.set_title(f"(b) Leak magnitude (n={N_SEEDS} seeds/point)", fontweight="bold")
ax.legend(frameon=False, fontsize=8)
ax.grid(alpha=0.3)

fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(OUT_DIR / "figures" / f"synthetic_leak_vs_alpha.{ext}",
                dpi=300, bbox_inches="tight")
plt.show()

## 7. Optional: fit an empirical leak(alpha) curve

A simple closed-form-ish fit (log-alpha vs leak) turns "we observed a trend"
into "we characterized the relationship" - useful for the paper's synthetic-
experiment section. Only fit on finite alphas (log(inf) undefined).

In [ ]:
fit_df = summ[np.isfinite(summ.alpha_num) & (summ.alpha_num < 1e8)].copy()
fit_df["log_alpha"] = np.log(fit_df["alpha_num"])

for (nc, ncl), g in fit_df.groupby(["n_classes", "n_clients"]):
    slope, intercept, r, p, se = sps.linregress(g["log_alpha"], g["leak_mean"])
    print(f"K={nc} C={ncl}: leak ~= {intercept:.4f} + ({slope:.4f}) * log(alpha)  "
          f"R^2={r**2:.3f}  p={p:.2e}")

## Notes for the paper

- `auc_pooled` sitting at ~0.5 across every alpha is the **control**: it
  proves the pooled protocol is leak-free by construction, purely a
  property of scoring every test row with one global constant.
- `auc_paired` rising as alpha decreases is the **effect**: it is produced
  entirely by the shared class-proportion matrix between train and test
  partitions, with *zero* image features, *zero* training, and *zero*
  dependence on GIED or endoscopy.
- The one-sample t-test in `synthetic_leak_significance.csv` is properly
  powered (n=100 seeds/cell) unlike the GIED-scale Wilcoxon test (n=5) -
  use this table, not the GIED seed test, as the paper's primary
  statistical evidence that the leak is real and not noise.
- Re-run with `N_CLASSES_GRID = [2, 5, 10]` and/or `N_CLIENTS_GRID = [5, 10, 20]`
  before the final draft to show the effect is not specific to GIED's
  5-class / 5-client configuration.